In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [ ]:
NAME = "Denver"
 
STATE = "08"

COUNTIES = [
    "031",  # Denver County
]

# NAME = "Los Angeles"

# STATE = "06"  # California

# COUNTIES = [
#     "037",  # Los Angeles County
#     "059",  # Orange County
#     "111"   # Ventura County
# ]

YEAR = 2022

API_KEY = "cf4fd28398432786abcfab6be3868157c4836d1d"

In [3]:
from src.acs import (
    load_acs_data,
    compute_acs_features
)

acs = load_acs_data(
    state_fips=STATE,
    county_fips_list=COUNTIES,
    year=YEAR,
    census_api_key=API_KEY
)

acs = compute_acs_features(acs)

In [4]:
acs = acs.drop(columns=["index"])
acs["state_fips"] = acs["tract_id"].str[:2]
acs["county_fips"] = acs["tract_id"].str[2:5]
acs["tract_code"] = acs["tract_id"].str[5:]

In [7]:
from src.geometry import load_tract_geometries

geometry = load_tract_geometries(
    state_fips=STATE,
    county_fips_list=COUNTIES
)

Loading tract geometries for state 08...
Raw tracts: 1447
Filtered tracts: 178
Geometry table ready.


In [11]:
print(geometry.columns.tolist())

['tract_id', 'state_fips', 'county_fips', 'tract_code', 'land_area_km2', 'centroid_lat', 'centroid_lon', 'geometry']


In [10]:
tracts = geometry.merge(
    acs,
    on="tract_id",
    how="left"
)

In [12]:
print(len(tracts))

print(
    tracts["tract_id"]
    .nunique()
)

print(
    tracts["total_population"]
    .isna()
    .sum()
)

178
178
0


In [16]:
from src.model_dataset import (
    build_modeling_dataset
)

model_df = build_modeling_dataset(
    tracts
)

In [19]:
print(model_df.columns.tolist())

['tract_id', 'total_population', 'population_density', 'pct_under_18', 'pct_18_64', 'pct_over_65', 'sex_ratio', 'median_age', 'avg_household_size', 'households', 'median_household_income', 'per_capita_income', 'poverty_rate', 'unemployment_rate', 'labor_force_participation_rate', 'snap_participation_rate', 'public_assistance_rate', 'pct_high_school', 'pct_bachelors_degree', 'pct_graduate_degree', 'pct_no_vehicle_households', 'pct_one_vehicle_households', 'pct_two_plus_vehicle_households', 'median_commute_time', 'pct_public_transit_commute', 'pct_car_commute', 'pct_walk_commute', 'pct_bike_commute', 'pct_work_from_home', 'housing_units', 'occupied_housing_units', 'vacant_housing_units', 'homeownership_rate', 'median_rent', 'median_home_value', 'housing_density', 'median_year_built', 'land_area_km2', 'pct_white_non_hispanic', 'pct_black', 'pct_hispanic_latino', 'pct_asian', 'pct_other_multiracial']


In [23]:
from pathlib import Path

output_dir = Path("../data/processed_predictors")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

model_df.to_parquet(
    output_dir / f"{NAME}_predictor_dataset.parquet",
    index=False
)